# 00 — legged_control の背景・目的・結論と全データフロー

## 1. リポジトリの背景
四足ロボットは、浮動胴体6自由度と12関節、接地/遊脚が切り替わるhybrid systemである。
関節PDだけでは、どの足で体重を支え、どの方向へ地面を蹴り、将来の接地切替に備えるかを
一貫して決めにくい。`qiayuanliao/legged_control` はこの問題を、

- OCS2のcentroidal NMPCによる **未来1秒の状態・GRF計画**
- qpOASESのWeighted WBCによる **現在瞬間の全身力学整合**
- 線形Kalman filterによる **浮動base並進推定**
- ros-controlによる **Gazebo/Unitree実機の共通I/O**

に分けた、A1/Go1/Aliengo向けのmodel-based locomotion baselineである。
公開元は2025年時点で開発終了を明記し、知覚統合の後継として `legged_perceptive` を案内している。

照合対象: [`legged_control` commit `a7f381c036`](https://github.com/qiayuanliao/legged_control/tree/a7f381c0367e98e31c01336e678eef47e304d40d)

> 重要: 上流C++は `external/legged_control/` に上記commitでclone済みだが、gitignore対象である。
> 主要経路はC++と `docs/legged_control/` の照合結果を使う。OCS2本体はこのworkspaceに無いため、
> OCS2内部の完全なODEは公開APIから分かる範囲、と区別する。

## 2. リポジトリの目的
上流READMEが掲げる目的は、NMPC・WBC・状態推定・sim2realを一つのROS制御stackとして提供し、
Unitree A1へ展開可能な高性能baselineにすることである。このNotebook系列の目的は別である。
完成stackをblack boxとして起動するのではなく、各境界を
**入力 → 数式 → C++処理 → 出力 → 次block** の順に読み直し、最終的にQ/R、摩擦、
WBC重み、周期、関節gainや制約式を変更できるようにする。

## 3. 先に結論
1. 速度/goalとGaitは別入力で、NMPCはGaitを選ばない。
2. NMPC状態・入力は各24次元。入力はGRF 12 + 関節速度12で、torqueではない。
3. NMPCは100 Hzで未来policyを作り、500 Hz側は現在時刻の1点だけを読む。
4. 既定WBCは階層QPではなくWeightedWbc単一QP。42変数の末尾torque 12だけを送る。
5. 関節指令はWBC torque feedforward + `Kp=0, Kd=3`。
6. 上流commit `a7f381c036` はROS1/OCS2の原実装である。一方、このprojectが所有する
   `src/legged_control_mujoco/` はMuJoCo実行adapterで、OCS2 SQPを実装していない。
7. Notebook 13は4秒の **equation-level proxy benchmark**、Notebook 14はadapterを実際に
   20秒以上動かすA1 MuJoCo benchmarkであり、どちらも上流ROS1/OCS2そのものの性能ではない。

## 4. 読む順序
1. 全体像とパッケージ
2. 状態・入力・座標系
3. 指令・参照・Gait
4. 状態推定
5. centroidal力学
6. NMPC
7. 接触制約
8. WBC
9. 関節ハイブリッド制御
10. 100 Hz / 500 Hz の統合
11. チューニングと数式変更
12. 実C++コードの端から端までのwalkthrough
13. equation-level proxy benchmark（上流性能ではない）
14. A1 MuJoCo adapterの30シナリオ実行証拠


In [1]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "pyproject.toml").exists():
        ROOT = candidate
        break

np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({"figure.figsize": (9, 4), "axes.grid": True})
print("repository:", ROOT)


repository: /home/takuya/work/mpc_dog


## 5. 一本の閉ループ — データの流れ

```text
人間/上位planner
  │
  ├─ /cmd_vel: [vx,vy,vz,yaw_rate] ──→ TargetTrajectoriesPublisher
  │                                      │ time(2), x_ref(2,24), u_ref(2,24)
  └─ gait名: stance/trot/... ─────────→ GaitReceiver
                                         │ ModeSchedule
                                         ▼
┌─────────────────────────────────────────────────────────────┐
│ OCS2 SQP-NMPC thread                                   100 Hz│
│ x0(24) + reference + contact schedule                         │
│ → horizon 1.0 s の x*(t,24), u*(t,24), mode(t) policy         │
└──────────────────────────┬──────────────────────────────────┘
                           │ shared policy
IMU(orientation,ω,a)       ▼
joint q,dq ──→ Linear KF / rbd conversion ──→ x_meas(24)
contact(4)          │ rbdState(36)                 │
                    └──────────────┬────────────────┘
                                   ▼
┌─────────────────────────────────────────────────────────────┐
│ LeggedController::update                               500 Hz│
│ evaluatePolicy(now,x) → x*(24),u*(24),mode                  │
│ WeightedWbc → [qdd(18), Fc(12), tau(12)]                    │
│ SafetyChecker → setCommand(q*,dq*,Kp=0,Kd=3,ff=tau)         │
└──────────────────────────┬──────────────────────────────────┘
                           ▼
                 Gazebo / Unitree motors
                           │ q,dq,IMU,contact
                           └──────────────→ 推定へ戻る
```

ここで最も大切な分離は次の3つ。

- **GaitはNMPCが選ばない。** 接地時間割として外から与える。
- **NMPCは関節トルクを決めない。** 未来の状態・GRF・関節速度を決める。
- **WBCは未来を解かない。** 現在瞬間の全身力学を満たすトルクをQPで決める。


In [2]:
# 各境界の次元を、接続できる「型」として確認する。
dimensions = {
    "command_twist_used": 4,
    "nmpc_state": 24,
    "nmpc_input": 24,
    "rbd_state": 36,
    "wbc_decision": 42,
    "joint_torque": 12,
    "hybrid_command_scalars": 12 * 5,
}
assert dimensions["wbc_decision"] == 18 + 12 + 12
assert dimensions["nmpc_input"] == 12 + 12
dimensions


{'command_twist_used': 4,
 'nmpc_state': 24,
 'nmpc_input': 24,
 'rbd_state': 36,
 'wbc_decision': 42,
 'joint_torque': 12,
 'hybrid_command_scalars': 60}

## 6. blockごとの契約

| Block | 入力 | 主な式/処理 | 出力 |
|---|---|---|---|
| 参照 | cmd + 現在姿勢 | $p^+=p+RvT$ | 2点 $x^{{ref}}$ |
| Gait | gait名 + 時刻 | mode schedule | 接地flag |
| 推定 | IMU,q,dq,接地 | linear KF | rbd 36 → x 24 |
| NMPC | x0,ref,mode | centroidal OCP/SQP | policy x*,u*,mode |
| WBC | x*,u*,rbd,mode | constrained QP | qdd,Fc,tau |
| 関節 | q*,dq*,tau | torque FF + low-gain PD | motor command |
| Plant | motor command | rigid-body/contact dynamics | sensors |

## 7. このNotebookの到達確認
自分の言葉で答える。

1. NMPCの24入力のうち、前半12と後半12は何か。
2. WBCの42変数のうち、モータへ送るのはどこか。
3. 100 HzのNMPC解を500 Hz側はどう使うか。
4. Gaitと胴体速度指令を分ける理由は何か。

次: `01_packages_and_loop.ipynb`
